In [2]:

import kagglehub
import pandas as pd
import os

/Users/charles/Developer/memoirev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
path = kagglehub.dataset_download("xfkzujqjvx97n/football-datasets")
tables = {}
print("Path to dataset files:", path)
for root, dirs, files in os.walk(path):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), path))
        if f.endswith(".csv"):
            nom = os.path.splitext(f)[0]
            tables[nom] = pd.read_csv(os.path.join(root, f), low_memory=False)


Path to dataset files: /Users/charles/.cache/kagglehub/datasets/xfkzujqjvx97n/football-datasets/versions/2
player_profiles/player_profiles.csv
player_national_performances/player_national_performances.csv
player_performances/player_performances.csv
player_injuries/player_injuries.csv
team_children/team_children.csv
team_competitions_seasons/team_competitions_seasons.csv
player_latest_market_value/player_latest_market_value.csv
player_market_value/player_market_value.csv
player_teammates_played_with/player_teammates_played_with.csv
team_details/team_details.csv
transfer_history/transfer_history.csv


In [4]:
for nom, df in tables.items():
    print(f"\n=== {nom} ===")
    print(df.shape)
    print(df.columns.tolist())


=== player_profiles ===
(92671, 34)
['player_id', 'player_slug', 'player_name', 'player_image_url', 'name_in_home_country', 'date_of_birth', 'place_of_birth', 'country_of_birth', 'height', 'citizenship', 'is_eu', 'position', 'main_position', 'foot', 'current_club_id', 'current_club_name', 'joined', 'contract_expires', 'outfitter', 'social_media_url', 'player_agent_id', 'player_agent_name', 'contract_option', 'date_of_last_contract_extension', 'on_loan_from_club_id', 'on_loan_from_club_name', 'contract_there_expires', 'second_club_url', 'second_club_name', 'third_club_url', 'third_club_name', 'fourth_club_url', 'fourth_club_name', 'date_of_death']

=== player_national_performances ===
(92701, 9)
['player_id', 'team_id', 'matches', 'goals', 'shirt_number', 'debut', 'coach_id', 'debut_game_id', 'career_state']

=== player_performances ===
(1878719, 20)
['player_id', 'season_name', 'competition_id', 'competition_name', 'team_id', 'team_name', 'nb_in_group', 'nb_on_pitch', 'goals', 'assist

In [5]:
# Test de l'unicité des clés primaires
for t, k in [("player_profiles", "player_id"), ("team_details", "club_id"),
             ("player_latest_market_value", "player_id")]:
    print(f"{t}.{k} unique :", tables[t][k].is_unique)

player_profiles.player_id unique : True
team_details.club_id unique : True
player_latest_market_value.player_id unique : True


In [33]:
def check_lien(src, col, ref, ref_col):
    s, r = tables[src][col].dropna(), tables[ref][ref_col].dropna()
    return {"lien": f"{src}.{col} → {ref}.{ref_col}",
            "% distincts": round(pd.Series(s.unique()).isin(r).mean() * 100, 1),
            "% lignes": round(s.isin(r).mean() * 100, 1)}
pd.set_option("display.max_colwidth", None)
liens = [
    # Couverture : part des joueurs de player_profiles présents dans chaque table
    ("player_profiles", "player_id", "player_performances", "player_id"),
    ("player_profiles", "player_id", "transfer_history", "player_id"),
    ("player_profiles", "player_id", "player_market_value", "player_id"),
    ("player_profiles", "player_id", "player_latest_market_value", "player_id"),
    ("player_profiles", "player_id", "player_injuries", "player_id"),
    ("player_profiles", "player_id", "player_national_performances", "player_id"),
    ("player_profiles", "player_id", "player_teammates_played_with", "player_id"),
    ("player_profiles", "player_id", "player_teammates_played_with", "teammate_player_id"),
    # Liens vers les clubs 
    ("player_performances", "team_id", "team_details", "club_id"),
    ("transfer_history", "from_team_id", "team_details", "club_id"),
    ("transfer_history", "to_team_id", "team_details", "club_id"),
    ("player_profiles", "current_club_id", "team_details", "club_id"),
    ("team_children", "parent_team_id", "team_details", "club_id"),
    ("team_children", "child_team_id", "team_details", "club_id"),
    ("player_national_performances", "team_id", "team_details", "club_id"),
    ("team_competitions_seasons", "club_id", "team_details", "club_id"),
    # Liens vers les compétitions (sens inchangé)
    ("player_performances", "competition_id", "team_competitions_seasons", "competition_id"),
    ("team_details", "competition_id", "team_competitions_seasons", "competition_id"),
]
display(pd.DataFrame([check_lien(*l) for l in liens]))

,lien,% distincts,% lignes
0,player_profiles.player_id → player_performances.player_id,95.4,95.4
1,player_profiles.player_id → transfer_history.player_id,98.4,98.4
2,player_profiles.player_id → player_market_value.player_id,74.9,74.9
3,player_profiles.player_id → player_latest_market_value.player_id,74.9,74.9
4,player_profiles.player_id → player_injuries.player_id,37.3,37.3
5,player_profiles.player_id → player_national_performances.player_id,30.4,30.4
6,player_profiles.player_id → player_teammates_played_with.player_id,91.1,91.1
7,player_profiles.player_id → player_teammates_played_with.teammate_player_id,56.9,56.9
8,player_performances.team_id → team_details.club_id,16.0,76.3
9,transfer_history.from_team_id → team_details.club_id,5.1,48.0


**Le noyeau des données de base dont on aura besoin plus tard est solide.** Les quatres tables indispensables (`player_profiles`, `player_performances`, `player_injuries`, `player_market_value`) sont reliées par player_id à 100 %. Construire la base joueur × saison ne pose donc pas de problème d’intégrité.

**Les faiblesses portent sur les clubs.** Ce sont plus des variables de contexte.

In [32]:
joueurs = tables["player_profiles"]["player_id"]

satellites = {
    "player_performances": "player_id",
    "player_market_value": "player_id",
    "player_latest_market_value": "player_id",
    "player_injuries": "player_id",
    "transfer_history": "player_id",
    "player_national_performances": "player_id",
    "player_teammates_played_with": "player_id",
}

couverture = pd.DataFrame({
    "table": list(satellites),
    "% joueurs couverts": [round(joueurs.isin(tables[t][c]).mean() * 100, 1)
                           for t, c in satellites.items()],
}).sort_values("% joueurs couverts", ascending=False)
display(couverture)

# Couverture combinée des briques indispensables au sujet
a_perf  = joueurs.isin(tables["player_performances"]["player_id"])
a_value = joueurs.isin(tables["player_market_value"]["player_id"])
a_inj   = joueurs.isin(tables["player_injuries"]["player_id"])

print("Performances + valeur marchande :", round((a_perf & a_value).mean() * 100, 1), "%")
print("Performances + valeur + au moins une blessure :", round((a_perf & a_value & a_inj).mean() * 100, 1), "%")

,table,% joueurs couverts
4,transfer_history,98.4
0,player_performances,95.4
6,player_teammates_played_with,91.1
1,player_market_value,74.9
2,player_latest_market_value,74.9
3,player_injuries,37.3
5,player_national_performances,30.4


Performances + valeur marchande : 74.6 %
Performances + valeur + au moins une blessure : 36.4 %


In [30]:
noms = (tables["player_performances"][["team_id", "team_name"]]
        .drop_duplicates("team_id").set_index("team_id")["team_name"])

top_nat = tables["player_national_performances"]["team_id"].value_counts().head(10)
print(top_nat.to_frame("nb_lignes").join(noms))

         nb_lignes team_name
team_id                     
21746          839       NaN
21100          760       NaN
21426          754       NaN
23133          728       NaN
16371          681       NaN
23243          667       NaN
9067           649       NaN
5710           648       NaN
16373          643       NaN
31269          606       NaN


Aucune equipe de player_national_performances n'est référencée dans TEAM_DETAILS car il s'agit d'une équipe nationale et non d'une équipe de club.

### Intégrité des données :

- `player_profiles` : 100 % des player_id sont uniques et non nuls. Les variables utiles à la modélisation sont quasi complète : id, name, main_position, date_of_birth, nationality, current_club_id, current_club_name. D'autres variables plus contextuelles sont très imcompletes mais non necessaires à la modélisation.

- `team_details` : 100 % des team_id sont uniques et non nuls. L'identité des clubs (nom, pays) est toujours renseignée. Les informations de compétition (competition_id, nom, club_division) manquent conjointement pour 23 % des clubs. Comme la table ne donne qu’une seule ligne par club, le championnat par saison devra de toute façon être repris de `team_competitions_seasons`.

- `team_children` : La table est complète sans aucune valeur manquante. Cependant son usage reste secondaire : rattacher les équipes réserves à leur club parent.

- `team_competitions_seasons` : Les clés club_id et season_id sont complètes. Les informations de compétition manquent conjointement pour 16 % des couples club-saison. C’est la table de référence pour attribuer un championnat et une division à chaque club par saison.

### `player_profiles`

In [7]:
print("Proportion de valeurs manquantes par variables dans la table Player_profiles")
pd.DataFrame({
    "nb" : tables["player_profiles"].isna().sum(),
    "pct" : (tables["player_profiles"].isna().mean() * 100 ).round(2)
}).sort_values("pct")


Proportion de valeurs manquantes par variables dans la table Player_profiles


,nb,pct
player_id,0,0.00
player_slug,0,0.00
player_image_url,0,0.00
current_club_name,0,0.00
current_club_id,0,0.00
main_position,2,0.00
height,4,0.00
is_eu,0,0.00
position,2,0.00
citizenship,57,0.06


In [8]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_profiles"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_profiles"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_profiles"].columns[tables["player_profiles"].isna().all()])


Nombre de lignes avec au moins une valeurs manquante :  92671
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_details`

In [9]:
print(tables["team_details"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_details")
pd.DataFrame({
    "nb" : tables["team_details"].isna().sum(),
    "pct" : (tables["team_details"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(2175, 12)
Proportion de valeurs manquantes par variables dans la table team_details


,nb,pct
club_id,0,0.00
club_slug,0,0.00
club_name,0,0.00
logo_url,0,0.00
country_name,0,0.00
season_id,0,0.00
source_url,0,0.00
_last_modified_at,0,0.00
competition_id,506,23.26
competition_slug,506,23.26


In [10]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_details"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_details"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_details"].columns[tables["team_details"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  506
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_children`

In [11]:
print(tables["team_children"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_children")
pd.DataFrame({
    "nb" : tables["team_children"].isna().sum(),
    "pct" : (tables["team_children"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(7695, 5)
Proportion de valeurs manquantes par variables dans la table team_children


,nb,pct
parent_team_id,0,0.0
parent_team_name,0,0.0
child_team_id,0,0.0
child_team_name,0,0.0
_last_modified_at,0,0.0


In [12]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_children"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_children"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_children"].columns[tables["team_children"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  0
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_competitions_seasons`

In [13]:
print(tables["team_competitions_seasons"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_competitions_seasons")
pd.DataFrame({
    "nb" : tables["team_competitions_seasons"].isna().sum(),
    "pct" : (tables["team_competitions_seasons"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(196378, 7)
Proportion de valeurs manquantes par variables dans la table team_competitions_seasons


,nb,pct
club_id,0,0.00
team_name,0,0.00
season_id,0,0.00
_last_modified_at,0,0.00
competition_name,32189,16.39
competition_id,32189,16.39
club_division,32189,16.39


In [14]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_competitions_seasons"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_competitions_seasons"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_competitions_seasons"].columns[tables["team_competitions_seasons"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  32189
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `PLAYER_TEAMMATES_PLAYED_WITH`

In [15]:
print(tables["player_teammates_played_with"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_teammates_played_with")
pd.DataFrame({
    "nb" : tables["player_teammates_played_with"].isna().sum(),
    "pct" : (tables["player_teammates_played_with"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(1257342, 6)
Proportion de valeurs manquantes par variables dans la table player_teammates_played_with


,nb,pct
player_id,0,0.00
teammate_player_id,0,0.00
teammate_player_name,0,0.00
ppg_played_with,37080,2.95
minutes_played_with,804153,63.96
joint_goal_participation,1035177,82.33


In [16]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_teammates_played_with"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_teammates_played_with"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_teammates_played_with"].columns[tables["player_teammates_played_with"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  1232285
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.


In [17]:
print("Doublons stricts :", tables["player_teammates_played_with"].duplicated().sum())

Doublons stricts : 0


Dans player_teammates_played_with : le couple (player_id, teammate_id) est-il unique, ou un même coéquipier apparaît-il plusieurs fois (par exemple s'ils ont joué ensemble dans plusieurs clubs différents) ?

In [18]:
nb_doublons_paires = (
    tables["player_teammates_played_with"]
    .duplicated(subset=["player_id", "teammate_player_id"])
    .sum()
)
print("Nombre de paires (joueur, coéquipier) répétées :", nb_doublons_paires)

Nombre de paires (joueur, coéquipier) répétées : 0


### `PLAYER INJURY HISTORIES`

In [19]:
print(tables["player_injuries"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_injuries")
pd.DataFrame({
    "nb" : tables["player_injuries"].isna().sum(),
    "pct" : (tables["player_injuries"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(143195, 7)
Proportion de valeurs manquantes par variables dans la table player_injuries


,nb,pct
player_id,0,0.00
season_name,0,0.00
injury_reason,0,0.00
games_missed,0,0.00
from_date,22,0.02
days_missed,22,0.02
end_date,1523,1.06


In [20]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_injuries"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_injuries"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_injuries"].columns[tables["player_injuries"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  1545
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.


HYPER IMPORTANT DE S'ATTARDER LÀ DESSUS

In [21]:
print("Doublons stricts :", tables["player_injuries"].duplicated().sum())

Doublons stricts : 111


In [22]:
lignes_par_joueur = tables["player_injuries"]["player_id"].value_counts()
print("Statistiques du nombre de blessures par joueur :")
print(lignes_par_joueur.describe())

Statistiques du nombre de blessures par joueur :
count    34561.000000
mean         4.143254
std          3.954910
min          1.000000
25%          1.000000
50%          3.000000
75%          6.000000
max         15.000000
Name: count, dtype: float64


days_missed et games_missed ont-ils des valeurs négatives impossibles ?


In [23]:
print(tables["player_injuries"][["days_missed", "games_missed"]].describe())
print()
print(tables["player_injuries"]["injury_reason"].value_counts().head(10))

         days_missed   games_missed
count  143173.000000  143195.000000
mean       51.719291       6.628479
std       104.437179       9.468380
min         1.000000       0.000000
25%        10.000000       1.000000
50%        22.000000       3.000000
75%        53.000000       7.000000
max      8655.000000     208.000000

injury_reason
unknown injury            27028
Muscle injury              6433
Hamstring injury           5786
Knee injury                5544
muscular problems          4880
Ankle injury               4602
Cruciate ligament tear     4035
Ill                        3694
Corona virus               3469
Knock                      2991
Name: count, dtype: int64


### `PLAYER_NATIONAL_TEAM_PERFORMANCES`

In [24]:
print(tables["player_national_performances"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_national_performances")
pd.DataFrame({
    "nb" : tables["player_national_performances"].isna().sum(),
    "pct" : (tables["player_national_performances"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(92701, 9)
Proportion de valeurs manquantes par variables dans la table player_national_performances


,nb,pct
player_id,0,0.00
team_id,0,0.00
matches,0,0.00
goals,0,0.00
shirt_number,0,0.00
career_state,0,0.00
debut_game_id,30347,32.74
coach_id,36922,39.83
debut,92701,100.00


In [25]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_national_performances"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_national_performances"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_national_performances"].columns[tables["player_national_performances"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  92701
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index(['debut'], dtype='str')


vérifier si des lignes ont été scrapées ou exportées en double.

In [26]:
print("Doublons stricts :", tables["player_national_performances"].duplicated().sum())

Doublons stricts : 32


Les statistiques de matchs, buts, ont-elles des anomalies (ex: but negatif?)

In [27]:
print(
    tables["player_national_performances"][
        ["matches", "goals"]
    ].describe()
)

            matches         goals
count  92701.000000  92701.000000
mean       7.336027      0.911382
std       12.943084      2.910156
min        0.000000      0.000000
25%        1.000000      0.000000
50%        4.000000      0.000000
75%        8.000000      1.000000
max      223.000000    141.000000
